# Reviewer Quickstart

This notebook inspects the bundled reviewer assets without requiring the full benchmark HDF5 files or model checkpoints.

It is intended to answer three quick questions:

1. What compact artifacts are bundled for reviewers?
2. What do the precomputed `RRUFF-325` examples look like?
3. How do the two example traces compare visually?


In [ ]:
from __future__ import annotations

import csv
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')

REVIEWER_DIR = REPO_ROOT / 'assets' / 'reviewer_examples'
REVIEWER_RESULTS = REPO_ROOT / 'results' / 'reviewer'

MANIFEST_PATH = REVIEWER_DIR / 'manifest.json'
METADATA_PATH = REVIEWER_DIR / 'reviewer_case_metadata.csv'
PRECOMPUTED_PATH = REVIEWER_RESULTS / 'rruff325_precomputed_inference.json'
PRIORS_PATH = REVIEWER_RESULTS / 'ext_group_priors.json'

for path in [MANIFEST_PATH, METADATA_PATH, PRECOMPUTED_PATH, PRIORS_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

manifest = json.loads(MANIFEST_PATH.read_text())
precomputed = json.loads(PRECOMPUTED_PATH.read_text())
priors = json.loads(PRIORS_PATH.read_text())
with METADATA_PATH.open() as handle:
    metadata_rows = list(csv.DictReader(handle))

print('Repo root:', REPO_ROOT)
print('Bundled reviewer examples:', len(manifest['examples']))
print('Precomputed inference cases:', len(precomputed['examples']))
print('Prior rows:', len(priors['rows']))
print('Reviewer metadata rows:', len(metadata_rows))


In [ ]:
top1_correct = sum(int(ex['top1_eg']) == int(ex['true_eg']) for ex in precomputed['examples'])
top1_accuracy = top1_correct / len(precomputed['examples'])
severity_counts = Counter(ex['severity'] for ex in precomputed['examples'])
fit_bucket_counts = Counter(ex['fit_bucket'] for ex in precomputed['examples'])

print('Checkpoint:', precomputed['checkpoint'])
print('Config:', precomputed['config'])
print('Benchmark basename:', manifest['benchmark_h5_basename'])
print(f'Top-1 accuracy from bundled examples: {100 * top1_accuracy:.2f}%')
print('Severity counts:', dict(severity_counts))
print('Fit-bucket counts:', dict(fit_bucket_counts))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(list(severity_counts.keys()), list(severity_counts.values()), color='steelblue')
axes[0].set_title('RRUFF-325 severity counts')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(list(fit_bucket_counts.keys()), list(fit_bucket_counts.values()), color='darkorange')
axes[1].set_title('Classical fit buckets')
axes[1].tick_params(axis='x', rotation=30)

fig.tight_layout()
plt.show()


In [ ]:
def load_trace(csv_path: Path):
    xs, ys = [], []
    with csv_path.open() as handle:
        reader = csv.reader(handle)
        header = next(reader)
        for row in reader:
            if not row:
                continue
            xs.append(float(row[0]))
            ys.append(float(row[1]))
    return xs, ys

correct_case = REVIEWER_DIR / 'correct_case_015_Arsenopyrite__R050071-1__6130.csv'
failure_case = REVIEWER_DIR / 'failure_case_000_Actinolite__R050336-1__5330.csv'
correct_meta = json.loads((REVIEWER_DIR / 'correct_case_015_Arsenopyrite__R050071-1__6130.json').read_text())
failure_meta = json.loads((REVIEWER_DIR / 'failure_case_000_Actinolite__R050336-1__5330.json').read_text())

xc, yc = load_trace(correct_case)
xf, yf = load_trace(failure_case)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(xc, yc, lw=1.0)
axes[0].set_title(f"Correct-case example: {correct_meta['case_id']} | true EG={correct_meta['true_eg']} | predicted EG={correct_meta['top1_eg']}")
axes[0].set_ylabel('Intensity')

axes[1].plot(xf, yf, lw=1.0, color='crimson')
axes[1].set_title(f"Failure-case example: {failure_meta['case_id']} | true EG={failure_meta['true_eg']} | predicted EG={failure_meta['top1_eg']}")
axes[1].set_xlabel(r'2$\\theta$')
axes[1].set_ylabel('Intensity')

fig.tight_layout()
plt.show()

print('Correct-case top-5 EGs:', correct_meta['top5_eg'])
print('Failure-case top-5 EGs:', failure_meta['top5_eg'])
